In [16]:
from multidata import speaker_traits, diarize
import os

In [8]:
os.chdir("/Users/kxw680/projects/multidata-local")
audio_path = "data/audio"
annotation_path = "data/diarization"

In [10]:
cases_with_audio = os.listdir(audio_path)
cases_with_annotations = os.listdir(annotation_path)
cases_to_process = list(set(cases_with_audio) & set(cases_with_annotations))

In [ ]:
def process_case(case_id):
    audio_file = os.listdir(os.path.join(audio_path, case_id))[0]
    annotation_file = os.listdir(os.path.join(annotation_path, case_id))[0]
    audio_file_path = os.path.join(audio_path, case_id, audio_file)
    annotation_file_path = os.path.join(annotation_path, case_id, annotation_file)
    annotation = diarize.load_rttm(annotation_file_path)
    speaker_stats = speaker_traits.speaker_pitch_stats(
        audio_file_path,
        annotation
    )
    for speaker, stats in speaker_stats.items():
        speaker_stats[speaker]['pitch_bucket'] = speaker_traits.pitch_bucket(stats['median_hz'])
    return speaker_stats

In [26]:
case_stats = {}
for case_id in cases_to_process:
    print(f"Processing case {case_id}...")
    case_stats[case_id] = process_case(case_id)


Processing case 261508...
Processing case 255053...
Processing case 261464...
Processing case 254901...
Processing case 261498...
Processing case 252848...
Processing case 260658...
Processing case 255050...
Processing case 261041...
Processing case 260660...
Processing case 255342...
Processing case 252850...
Processing case 252851...
Processing case 261456...
Processing case 252849...
Processing case 259392...
Processing case 255387...
Processing case 261485...
Processing case 261573...
Processing case 261505...
Processing case 261268...
Processing case 261488...
Processing case 254921...
Processing case 255048...
Processing case 260656...
Processing case 255098...
Processing case 261473...
Processing case 261320...
Processing case 254867...
Processing case 255386...
Processing case 260723...
Processing case 261238...


In [27]:
case_stats

{'261508': {'SPEAKER_01': {'median_hz': 125.9055451532844,
   'n_voiced_frames': 9203,
   'pitch_bucket': 'lower'},
  'SPEAKER_00': {'median_hz': 188.87446692169735,
   'n_voiced_frames': 10292,
   'pitch_bucket': 'higher'}},
 '255053': {'SPEAKER_02': {'median_hz': 224.4654454301898,
   'n_voiced_frames': 404,
   'pitch_bucket': 'higher'},
  'SPEAKER_03': {'median_hz': 222.40169155960237,
   'n_voiced_frames': 8471,
   'pitch_bucket': 'higher'},
  'SPEAKER_00': {'median_hz': 199.68449743614545,
   'n_voiced_frames': 16191,
   'pitch_bucket': 'higher'},
  'SPEAKER_01': {'median_hz': 220.07075070457464,
   'n_voiced_frames': 1617,
   'pitch_bucket': 'higher'}},
 '261464': {'SPEAKER_01': {'median_hz': 212.37298381618444,
   'n_voiced_frames': 3483,
   'pitch_bucket': 'higher'},
  'SPEAKER_03': {'median_hz': 178.396037267192,
   'n_voiced_frames': 5848,
   'pitch_bucket': 'ambiguous'},
  'SPEAKER_02': {'median_hz': 111.3342636783288,
   'n_voiced_frames': 44748,
   'pitch_bucket': 'lower'}

In [33]:
normalized = []
for case_id, speakers in case_stats.items():
    for speaker, stats in speakers.items():
        normalized.append({
            'case_id': case_id,
            'speaker': speaker,
            'median_hz': stats['median_hz'],
            'n_voiced_frames': stats['n_voiced_frames'],
            'pitch_bucket': stats['pitch_bucket']
        })

normalized = pd.DataFrame(normalized)

In [34]:
normalized.groupby(['case_id','pitch_bucket']).size()

case_id  pitch_bucket
252848   ambiguous       2
         lower           6
252849   ambiguous       2
         lower           3
252850   higher          1
         lower           2
252851   lower           2
254867   higher          4
         lower           1
254901   higher          5
254921   higher          2
         lower           1
255048   higher          4
255050   higher          2
         lower           1
255053   higher          4
255098   higher          2
         lower           1
255342   higher          4
         lower           1
255386   ambiguous       4
         higher          1
         lower           3
255387   lower           2
259392   higher          1
         lower           4
260656   lower           1
260658   ambiguous       1
         higher          2
         lower           2
260660   ambiguous       2
         higher          4
         lower           2
260723   ambiguous       2
         lower           2
261041   higher          2
261238

In [35]:
normalized.to_csv("data/speaker_pitch_stats.csv", index=False)